# L1 · Primer contacto — Sesión 1: **Recibir**
### Introducción a las Ciencias de la Computación · UNAL Manizales

El 16 de noviembre de 1974, el radiotelescopio de Arecibo, en Puerto Rico, mandó hacia el cúmulo de estrellas M13 un mensaje de **1 679 bits**. Sin idioma, sin instrucciones, sin un "léase así". Solo ceros y unos.

Hoy ustedes son los extraterrestres que lo reciben.

**Cómo se trabaja en este cuaderno**

- Las preguntas de **Momento 1** se responden **en papel, antes de tocar el computador**. Tómenle foto y súbanla al final del cuaderno. No hay predicción mala: hay predicción no escrita.
- Cada herramienta es una **función**, y cada función se escribe **primero su contrato** (qué recibe, qué devuelve, qué supone) y **después** su código.
- Debajo de cada función hay una celda de **pruebas** con `assert`. Si una prueba falla, la celda se queja con un `AssertionError`. Eso no es un regaño: es la máquina diciéndoles exactamente dónde mirar.
- Las celdas que dicen **CAJA** ya vienen hechas. Pueden abrirlas, pero no hace falta: basta con su contrato. Así usan todos los días `print()` o `range()`.
- Donde dice `# TU CÓDIGO AQUÍ`, escriben ustedes. Donde dice ✍️, responden en texto.

In [ ]:
# Integrantes del equipo (nombre completo)
EQUIPO = ["", "", "", ""]

## Momento 1 — Predecir *(en papel, 15 minutos)*

**P1.** Les llega una cadena de 1 679 ceros y unos, sin nada más. ¿Qué es lo primero que intentarían para encontrarle sentido? Escriban tres ideas, en orden.

**P2.** Si alguien quisiera mandar un **dibujo** con bits, ¿cómo lo haría? ¿Qué tendría que adivinar quien lo recibe?

**P3.** ¿De cuántas maneras se pueden acomodar 1 679 cuadritos en un rectángulo completo, sin que sobre ni falte ninguno? Sin computador: con lo que saben de divisores.

✍️ *Copien aquí sus respuestas cortas (la foto del papel va al final):*

- P1:
- P2:
- P3:

## 0. Traer el mensaje

La celda que sigue es una **caja**. Su contrato es una línea: `cargar_mensaje()` devuelve una cadena de 1 679 caracteres, cada uno `'0'` o `'1'`. Si ya tienen el archivo `datos/arecibo.txt`, lo lee de ahí; si no, lo descarga de la copia congelada del curso o de la fuente original.

También trae `huella(bits)`, que devuelve cuántos unos tiene el mensaje y su **SHA-256**: una "huella digital" de 64 caracteres. Si cambia un solo bit del mensaje, la huella cambia por completo. Es la misma idea con la que su banco sabe que nadie alteró una transferencia en el camino.

In [ ]:
# ==========================================================================
#  CAJA: traer el mensaje de Arecibo.  No hace falta abrirla; sí entender su contrato.
#  cargar_mensaje() -> str   Devuelve los 1679 caracteres '0'/'1' del mensaje.
#  Busca, en este orden: (1) el archivo datos/arecibo.txt si ya lo tienen,
#  (2) la copia congelada del repositorio de laboratorios, (3) la fuente original (Wikipedia).
# ==========================================================================
import os, re, html, hashlib, urllib.request

REPO_RAW = "https://raw.githubusercontent.com/cmorregof/labs-calculo-2026-2/main"
LONGITUD = 1679

def _descargar(url):
    pedido = urllib.request.Request(url, headers={"User-Agent": "UNAL-IntroCC-L1/1.0"})
    with urllib.request.urlopen(pedido, timeout=30) as r:
        return r.read().decode("utf-8", errors="replace")

def _extraer_bits(texto):
    if re.search(r"<tr\b", texto, flags=re.I):
        filas = [re.sub(r"\s", "", html.unescape(re.sub(r"<[^>]+>", "", "<" + re.split(r"</tr>", t, flags=re.I)[0])))
                 for t in re.split(r"<tr\b", texto, flags=re.I)[1:]]
        actual = []
        for f in filas + ["x"]:
            if f and set(f) <= {"0", "1"}:
                actual.append(f)
            else:
                if len("".join(actual)) == LONGITUD:
                    return "".join(actual)
                actual = []
    texto = html.unescape(re.sub(r"<[^>]+>", "\n", texto))
    bloques, actual = [], []
    for renglon in texto.splitlines():
        limpio = re.sub(r"\s", "", renglon)
        if limpio == "":
            continue
        if set(limpio) <= {"0", "1"}:
            actual.append(limpio)
        else:
            if actual:
                bloques.append("".join(actual))
            actual = []
    if actual:
        bloques.append("".join(actual))
    for b in bloques:
        if len(b) == LONGITUD:
            return b
    return None

def cargar_mensaje():
    for local in ("datos/arecibo.txt", "../datos/arecibo.txt"):
        if os.path.exists(local):
            with open(local) as f:
                print("Mensaje leído de", local)
                return f.read().strip()
    intentos = [REPO_RAW + "/datos/arecibo.txt",
                "https://en.wikipedia.org/w/index.php?title=Arecibo_message&action=raw",
                 "https://en.wikipedia.org/wiki/Arecibo_message"]
    for url in intentos:
        try:
            bits = _extraer_bits(_descargar(url))
            if bits:
                os.makedirs("datos", exist_ok=True)
                with open("datos/arecibo.txt", "w") as f:
                    f.write(bits)
                print("Mensaje descargado de", url)
                return bits
        except Exception as e:
            print("No se pudo con", url, "->", e)
    raise RuntimeError("No pude traer el mensaje. Avísenle al monitor: hay una copia de respaldo.")

def huella(bits):
    """Devuelve (número de unos, SHA-256). Si cambia un solo bit, el SHA-256 cambia por completo."""
    return bits.count("1"), hashlib.sha256(bits.encode("ascii")).hexdigest()

In [ ]:
bits = cargar_mensaje()
unos, sha = huella(bits)
print("Unos:", unos)
print("Huella SHA-256:", sha)
print("Comparen estas dos cifras con las del README de introduccion_cc en el repositorio de laboratorios.")

## 1. Conocer el dato

Antes de calcular nada, se mira. Un dato que no se miró antes de usarlo es un dato en el que se confió sin razón. Contesten **con código**:

- ¿Cuántos caracteres tiene `bits`? (`len`)
- ¿Qué caracteres **distintos** aparecen? (`set`)
- ¿Cuántos unos y cuántos ceros hay? (`.count`)
- Miren los primeros 60 caracteres. ¿Ven algún patrón a simple vista?

In [ ]:
n_caracteres = None        # TU CÓDIGO AQUÍ
distintos = None           # TU CÓDIGO AQUÍ
n_unos = None              # TU CÓDIGO AQUÍ
n_ceros = None             # TU CÓDIGO AQUÍ

print("caracteres:", n_caracteres)
print("distintos :", distintos)
print("unos/ceros:", n_unos, n_ceros)
print(bits[:60])

In [ ]:
# Pruebas
assert n_caracteres == 1679, "¿Seguro que contaron bien? El mensaje tiene 1679 bits."
assert distintos == {"0", "1"}, "Debería haber solo ceros y unos."
assert n_unos + n_ceros == n_caracteres
print("✔ Conocen el dato.")

**Una ironía para pensar.** En el archivo, cada bit del mensaje está guardado como un **carácter**, y en el disco un carácter ocupa 8 bits. Corran la celda: ¿cuántos bits ocupa en su computador el mensaje de 1 679 bits? Representar algo siempre cuesta algo, y lo que cuesta depende de la convención que se eligió.

In [ ]:
bits_en_disco = len(bits.encode("ascii")) * 8
print("El mensaje tiene", len(bits), "bits de información y ocupa", bits_en_disco, "bits en el disco.")

✍️ *¿Cuál es la diferencia entre el carácter `'1'` y el bit 1? Dos líneas.*

## 2. La llave: `es_primo(n)`

En la clase 5 escribieron esto en PSeInt:

```
esPrimo <- (n >= 2);
d <- 2;
Mientras d * d <= n y esPrimo Hacer
    Si n mod d = 0 Entonces
        esPrimo <- Falso;
    FinSi
    d <- d + 1;
FinMientras
```

Tradúzcanlo a Python **como función**. Recuerden el diccionario: `mod` es `%`, `y` es `and`, `Falso` es `False`, y lo que va dentro del `while` lo dice la sangría. La función **devuelve** el resultado con `return`; no lo imprime.

In [ ]:
def es_primo(n):
    """CONTRATO
    Recibe: un entero n.
    Devuelve: True si n es primo, False si no.
    Supone: n >= 0.
    """
    # TU CÓDIGO AQUÍ
    pass

In [ ]:
# Pruebas
assert es_primo(2) and es_primo(3) and es_primo(97) and es_primo(1000003)
assert not es_primo(0) and not es_primo(1) and not es_primo(91) and not es_primo(1000001)
print("✔ es_primo pasa las pruebas.")
print("¿1679 es primo?", es_primo(1679))

## 3. Las formas posibles: `rectangulos(n)`

Un rectángulo de `filas × columnas` cuadritos usa exactamente `n` cuadritos cuando `filas * columnas == n`. La función devuelve **todas** las parejas posibles, ordenadas por número de filas.

Ejemplo: `rectangulos(12)` debe devolver `[(1, 12), (2, 6), (3, 4), (4, 3), (6, 2), (12, 1)]`.

In [ ]:
def rectangulos(n):
    """CONTRATO
    Recibe: un entero n >= 1.
    Devuelve: la lista de parejas (filas, columnas) con filas * columnas == n,
              ordenada de menor a mayor número de filas.
    """
    # TU CÓDIGO AQUÍ
    pass

In [ ]:
# Pruebas
assert rectangulos(12) == [(1, 12), (2, 6), (3, 4), (4, 3), (6, 2), (12, 1)]
assert rectangulos(7) == [(1, 7), (7, 1)]
assert len(rectangulos(1680)) == 40
print("✔ rectangulos pasa las pruebas.")
print("Formas de acomodar 1679:", rectangulos(1679))
print("Formas de acomodar 1680:", len(rectangulos(1680)))

✍️ **La pregunta de fondo** (va al informe). ¿Por qué quien diseñó el mensaje eligió un número que es producto de **dos primos**? ¿Qué habría pasado si el mensaje tuviera 1 680 bits, que se pueden acomodar de 40 maneras?

**Para quien termine antes: mejorar el algoritmo.** Su `rectangulos` probablemente prueba todos los números de 1 a `n`. Con la idea de la clase 5 basta probar hasta $\sqrt{n}$: cada divisor `d` pequeño trae de regalo a su pareja `n // d`. Escriban `rectangulos_rapido(n)`, cuenten las vueltas de cada versión con `n = 1679` y con `n = 1 000 000`, y comprueben que las dos devuelven lo mismo.

In [ ]:
# (opcional) rectangulos_rapido(n) y conteo de vueltas

## 4. Ver: `dibujar(bits, columnas)`

Ahora sí: acomodar los bits en filas y mirarlos. Lo hacemos en dos funciones, porque una función que **devuelve** se puede probar, y una que solo **imprime**, no.

- `filas_de(bits, columnas)` devuelve la lista de renglones, cada uno una cadena de `columnas` caracteres. Ejemplo: `filas_de("101100", 3)` es `["101", "100"]`.
- `dibujar(bits, columnas)` usa `filas_de` e imprime cada renglón cambiando `"1"` por `"█"` y `"0"` por un espacio.

In [ ]:
def filas_de(bits, columnas):
    """CONTRATO
    Recibe: una cadena de bits y un número de columnas que divide su longitud.
    Devuelve: la lista de renglones, cada uno de longitud 'columnas'.
    """
    # TU CÓDIGO AQUÍ
    pass


def dibujar(bits, columnas):
    """CONTRATO
    Recibe: una cadena de bits y un número de columnas.
    Devuelve: nada. Imprime los renglones con '█' para 1 y ' ' para 0.
    """
    # TU CÓDIGO AQUÍ (usen filas_de)
    pass

In [ ]:
# Pruebas
assert filas_de("101100", 3) == ["101", "100"]
assert filas_de("101100", 2) == ["10", "11", "00"]
assert len(filas_de(bits, 23)) == 73
print("✔ filas_de pasa las pruebas.")

Prueben las dos formas que encontraron. **Una de las dos es ruido; la otra no.** Este es el momento de dejarlo correr y esperar a ver si coincide con lo que pensaban.

In [ ]:
dibujar(bits, 73)

In [ ]:
dibujar(bits, 23)

La caja que sigue dibuja lo mismo como imagen, con la cuadrícula numerada. Les va a servir para ubicar filas y columnas en el paso 5.

In [ ]:
# CAJA: dibujo con cuadrícula numerada, para encontrar filas y columnas.
# dibujar_imagen(bits, columnas) -> None   Muestra los bits como imagen: 1 = negro, 0 = blanco.
import matplotlib.pyplot as plt

def dibujar_imagen(bits, columnas, alto=12):
    filas = len(bits) // columnas
    matriz = [[int(bits[f * columnas + c]) for c in range(columnas)] for f in range(filas)]
    fig, ax = plt.subplots(figsize=(alto * columnas / max(filas, 1) + 1, alto))
    ax.imshow(matriz, cmap="Greys", interpolation="nearest")
    ax.set_xticks(range(0, columnas, 5)); ax.set_yticks(range(0, filas, 5))
    ax.set_xticks([x - 0.5 for x in range(columnas + 1)], minor=True)
    ax.set_yticks([y - 0.5 for y in range(filas + 1)], minor=True)
    ax.grid(which="minor", color="#dddddd", linewidth=0.4)
    ax.set_xlabel("columna"); ax.set_ylabel("fila")
    ax.set_title(f"{filas} filas x {columnas} columnas")
    plt.show()

In [ ]:
dibujar_imagen(bits, 23)

## 5. Leer: `leer_numero(columna)`

En la parte de arriba del dibujo hay números escritos en **binario**. Recordemos cómo se lee un número en base 2: cada posición vale el doble de la que tiene a la derecha.

$$1011_2 = 1\cdot 8 + 0\cdot 4 + 1\cdot 2 + 1\cdot 1 = 11$$

Hay una manera de leerlo con un ciclo y sin potencias, de izquierda a derecha: arrancar en 0 y, por cada bit, **duplicar lo que llevo y sumarle el bit**. Con `1011`: 0 → 1 → 2 → 5 → 11. Es un acumulador, como el de Gauss. Hagan la tabla de seguimiento en papel antes de escribirla.

In [ ]:
def leer_numero(lista_bits):
    """CONTRATO
    Recibe: una lista de enteros 0 y 1; el primero es el más significativo.
    Devuelve: el entero que representan en base 2.
    """
    # TU CÓDIGO AQUÍ
    pass

In [ ]:
# Pruebas
assert leer_numero([1, 0, 1, 1]) == 11
assert leer_numero([0]) == 0 and leer_numero([1]) == 1
assert leer_numero([1, 0, 0, 0, 0, 0, 0, 0]) == 128
assert leer_numero([0, 0, 1, 1]) == 3
print("✔ leer_numero pasa las pruebas.")

Los números del mensaje están escritos **en vertical**, en columnas. Hace falta una herramienta para sacar un pedazo de columna de la cuadrícula.

In [ ]:
def columna(bits, columnas, j, fila_desde, fila_hasta):
    """CONTRATO
    Recibe: los bits, el ancho de la cuadrícula, una columna j y un rango de filas.
    Devuelve: la lista de enteros (0/1) de la columna j, desde fila_desde hasta fila_hasta - 1,
              de arriba hacia abajo.
    """
    # TU CÓDIGO AQUÍ
    pass

In [ ]:
# Pruebas (una cuadrícula de juguete de 3 x 4)
juguete = "1000" + "0101" + "1101"
assert columna(juguete, 4, 0, 0, 3) == [1, 0, 1]
assert columna(juguete, 4, 3, 1, 3) == [1, 1]
print("✔ columna pasa las pruebas.")

Y una caja para mirar de cerca una región, con los números de fila y de columna al borde:

In [ ]:
# CAJA: mirar de cerca una región.  recorte(bits, columnas, f0, f1, c0, c1) imprime
# las filas f0..f1-1 y las columnas c0..c1-1, con los números de fila y de columna al borde.
def recorte(bits, columnas, f0, f1, c0, c1):
    print("     " + "".join(str(c % 10) for c in range(c0, c1)))
    for f in range(f0, f1):
        fila = bits[f * columnas:(f + 1) * columnas]
        print(f"{f:3d}  " + "".join("█" if ch == "1" else "·" for ch in fila[c0:c1]))

In [ ]:
recorte(bits, 23, 0, 12, 0, 23)    # las primeras 12 filas, todas las columnas

### Descifrar

1. **La primera sección.** Ubiquen, en las primeras filas, las columnas que forman números. Extráiganlas con `columna(...)` y léanlas con `leer_numero(...)`. Hay **una sola dirección** de lectura (de arriba abajo, o de abajo arriba con `lista[::-1]`) que hace que la sección diga algo que reconocen. Encuéntrenla probando las dos.
   - Ojo: puede haber bits que **no** son parte del número, sino marcas que dicen "aquí hay un número". ¿Hay una fila con un 1 debajo de cada número?
   - Algunos números pueden ocupar más de una columna. ¿Cuáles, y por qué?
2. **La segunda sección.** Un poco más abajo hay cinco números más. Léanlos igual y búsquenlos en la tabla periódica de la celda siguiente.

In [ ]:
# Los primeros 20 elementos de la tabla periódica, por número atómico
TABLA_PERIODICA = {1: "H hidrógeno", 2: "He helio", 3: "Li litio", 4: "Be berilio", 5: "B boro",
                   6: "C carbono", 7: "N nitrógeno", 8: "O oxígeno", 9: "F flúor", 10: "Ne neón",
                   11: "Na sodio", 12: "Mg magnesio", 13: "Al aluminio", 14: "Si silicio",
                   15: "P fósforo", 16: "S azufre", 17: "Cl cloro", 18: "Ar argón",
                   19: "K potasio", 20: "Ca calcio"}

In [ ]:
# Su exploración. Ejemplo de uso (cambien los números de fila y columna):
# leer_numero(columna(bits, 23, j=1, fila_desde=1, fila_hasta=4))
# leer_numero(columna(bits, 23, j=1, fila_desde=1, fila_hasta=4)[::-1])

✍️ **Lo que leyeron:**

- Primera sección (filas ___ a ___): los números ______________ . Dirección de lectura: ______ , porque ______ .
- Segunda sección (filas ___ a ___): los números ______________ , que corresponden a ______________ . ¿De qué molécula de la vida son esos elementos?

## 6. Inventario

Con el dibujo y sus funciones, hagan la lista de lo que creen que dice cada parte del mensaje. **No hace falta descifrarlo todo**: hace falta separar lo que descifraron **con código** de lo que están adivinando **con los ojos**.

| Filas | Qué creemos que dice | ¿Lo leímos con código o lo adivinamos? | ¿Cómo lo comprobaríamos? |
|---|---|---|---|
| | | | |
| | | | |
| | | | |
| | | | |

## Momento 3 — Explicar *(media página por equipo)*

**La lista de convenciones.** El emisor supuso que el receptor ya sabía ciertas cosas. Escríbanlas todas las que encuentren, y para cada una decidan si un extraterrestre de verdad podría **deducirla** o tendría que **adivinarla**.

| Convención que supuso el emisor | ¿Se puede deducir o hay que adivinarla? | ¿Por qué? |
|---|---|---|
| Los bits forman un rectángulo | | |
| | | |
| | | |
| | | |

**Confrontación.** ¿Qué predijeron en P1, P2 y P3, qué encontraron y a qué atribuyen la diferencia?

**La frase obligatoria**, con sus palabras: *"Representar es..."*

✍️

## La IA como objeto de estudio *(para la bitácora individual)*

Denle a su asistente (Claude, o el que tengan) **la cadena de 1 679 bits sin decirle qué es**, y pídanle que la descifre. Guarden la respuesta completa en la bitácora.

¿La **descifró** (hizo cuentas que ustedes pueden verificar con sus funciones) o la **reconoció** (sabía de antemano qué era y recitó)? Propongan una prueba para distinguirlo. Pista: en la sesión 2 van a tener un mensaje que no está en ningún lugar de internet.

## Foto de las predicciones

Suban la foto del papel de Momento 1 (en Colab: el ícono de carpeta a la izquierda → subir) y pongan aquí su nombre de archivo:

`foto_predicciones_sesion1.jpg`

---
### Antes de irse
1. **Ejecuten todo de arriba a abajo** (*Entorno de ejecución → Reiniciar y ejecutar todo*). Si algo falla, no está terminado.
2. **Guarden una copia en su repositorio `intro-cc`:** *Archivo → Guardar una copia en GitHub*. Cada integrante en el suyo.
3. La **bitácora de IA** es individual y va aparte (la plantilla es `bitacora_IA_L1.md`, en la carpeta `introduccion_cc` del repositorio de laboratorios).